In [23]:
import datetime


def process_book(book_path):
    from propp_fr import load_models
    spacy_model, mentions_detection_model, coreference_resolution_model = load_models()

    ######################################################################################

    from propp_fr import load_text_file
    text_content = load_text_file("la_comedie_humaine/etudes_des_moeurs/scenes_de_la_vie_de_province/balzac-33-FC-rabouilleuse/balzac-33-FC-rabouilleuse.txt")

    ######################################################################################

    from propp_fr import generate_tokens_df
    tokens_df = generate_tokens_df(text_content, spacy_model)

    ######################################################################################

    from propp_fr import load_tokenizer_and_embedding_model, get_embedding_tensor_from_tokens_df

    # Load the tokenizer and pre-trained embedding model
    tokenizer, embedding_model = load_tokenizer_and_embedding_model(
        mentions_detection_model["base_model_name"],
      )

    # Generate embeddings for all tokens
    tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(
        text_content,
        tokens_df,
        tokenizer,
        embedding_model,
      )

    ######################################################################################

    from propp_fr import generate_entities_df

    entities_df = generate_entities_df(
        tokens_df,
        tokens_embedding_tensor,
        mentions_detection_model,
    )

    ######################################################################################

    from propp_fr import add_features_to_entities

    entities_df = add_features_to_entities(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import perform_coreference

    entities_df = perform_coreference(
        entities_df,
        tokens_embedding_tensor,
        coreference_resolution_model,
        )

    ######################################################################################

    from propp_fr import extract_attributes
    tokens_df = extract_attributes(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import generate_characters_dict

    characters_dict = generate_characters_dict(tokens_df, entities_df)

    ######################################################################################

    return tokens_df, entities_df, characters_dict

In [ ]:
import os
from propp_fr import save_tokens_df, save_entities_df, save_book_file


for subdir, dirs, files in os.walk("la_comedie_humaine"):
    for file in files:
        if file.endswith(".txt"):
            print("Processing file: " + file)
            print("Start time: " + str(datetime.datetime.now()))
            book_path = os.path.join(subdir, file)
            tokens, entities, characters = process_book(book_path)
            save_tokens_df(tokens, file[:-4]+"_tokens", subdir)
            save_entities_df(entities, file[:-4]+"_entities", subdir)
            save_book_file(characters, file[:-4]+"_book_file", subdir)
            print("End time: " + str(datetime.datetime.now()))